# Vanilla BERT + GPT Approach

This method has two steps involved:

1. Use Vanilla BERT to tag all sentences with temporal text.
2. Use GPT-5.4 mini to filter for time-specific sentences.

In [ ]:
%pip install transformers openai tqdm torch nltk python-dotenv -q

## Imports & Config

In [ ]:
import json
import os
import numpy as np
from dotenv import load_dotenv
from transformers import pipeline
from openai import OpenAI
from tqdm import tqdm

load_dotenv()

# bypass SSL issue
# see: https://github.com/gunthercox/ChatterBot/issues/930#issuecomment-322111087
import nltk
import ssl
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import sent_tokenize

# Config for the models
BOOK_FILE = "moby_dick.txt"
BERT_OUTPUT_FILE = "bert_candidates.json" # GPT-5.4 will filter this output
OUTPUT_FILE = "gpt_filtered.json" # filtered output written here
TIMEX3_ATTRIBUTES = {"DURATION", "DATE", "TIME", "WEEKDATE", "WEEKTIME", "SEASON", "PARTOFYEAR", "PAPRFU"}

CONTEXT_WINDOW = 1 # sentences of context around each hit
GPT_MODEL = "gpt-5.4-mini"
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

## Step 1: Extract temporal sentences with BERT token classifier


In [ ]:
def extract_temporal_candidates(filepath, context_window=1):
    """Return sentences that contain a TIMEX3 TIME or DATE entity."""
    print("Loading BERT Temporal Tagger...")
    tagger = pipeline(
        "token-classification",
        model="satyaalmasian/temporal_tagger_BERT_tokenclassifier",
        aggregation_strategy="simple",
    )

    print(f"Reading: {filepath}")
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    # use NLTK to split the book up into sentences
    sentences = sent_tokenize(text.replace("\n", " "))
    sentences = [s.strip() for s in sentences if s.strip()]

    # iterate over all of the book's sentences and have
    # the BERT model tag any with time-related text.
    candidates = []
    print("\nTagging sentences...")
    for i, sentence in enumerate(tqdm(sentences, desc="BERT tagging")):
        entities = tagger(sentence)
        
        if any(e["entity_group"] in TIMEX3_ATTRIBUTES for e in entities):
            start = max(0, i - context_window)
            end = min(len(sentences), i + context_window + 1)
            candidates.append({
                "sentence_index": i,
                "target_sentence": sentence,
                "full_context": " ".join(sentences[start:end]),
                "timex3_entities": [
                    {"type": e["entity_group"], "text": e["word"], "score": round(float(e["score"]), 4)}
                    for e in entities
                ],
            })

    print(f"\nFound {len(candidates)} temporal candidates.")
    return candidates

In [ ]:
'''
Tag the text with Vanilla BERT, and write
the tagged sentences to a JSON file.
'''
candidates = extract_temporal_candidates(BOOK_FILE, context_window=CONTEXT_WINDOW)

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.integer):
            return int(obj)
        return super().default(obj)

with open(BERT_OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(candidates, f, indent=4, ensure_ascii=False, cls=NumpyEncoder)

print(f"Saved {len(candidates)} Vanilla BERT candidates to {BERT_OUTPUT_FILE}")

## Step 2: Filter via GPT-5.4 mini

For each candidate input, GPT determines:
1. Whether a specific clock time is present
2. The minimal excerpt that includes the clock reference with enough surrounding context to be meaningful. This may just be the clock sentence, or it may include  neighboring sentences if for context or coherence.

Returns `NONE` if no clock time is present.

In [ ]:
SYSTEM_PROMPT = """\
You are a literary analyst extracting clock-based time references from a novel.

Given a passage, do the following:
1. Determine whether the passage contains a specific clock time — a precise time \
of day that could appear on a clock face (e.g. "nine o'clock", "2 o'clock", \
"4:30 p.m.", "noon", "midnight", "half past three", "quarter to four", \
"almost two"). Vague time-of-day words (morning, afternoon, evening, night, \
tonight, dawn, dusk, twilight), relative time (yesterday, last summer, tomorrow \
morning), durations (for two hours), meal times (dinner time), idioms \
(at the last minute), and salutations (Good night) do NOT count.
2. If a clock time is present, return the minimal excerpt from the passage that \
includes the clock reference AND enough surrounding context for the quote to make \
sense on its own. Include a preceding sentence if the clock sentence is a \
continuation that would be unclear without it (e.g. a note whose content is \
on the next line). Include both the preceding and the following sentence, the \
preceding sentence, or the following sentence if one or both of them meaningfully \
enrich the clock reference. If the clock sentence stands alone, return only that \
sentence.
3. If no clock time is present, return exactly: NONE

Return ONLY the excerpt text or NONE. Do not explain your reasoning.\
"""

def extract_clock_quote(item: dict) -> str | None:
    """Return the best excerpt containing a clock time, or None if absent."""
    tagged = ", ".join(f'"{e["text"]}" ({e["type"]})' for e in item["timex3_entities"])
    user_msg = f'Passage:\n{item["full_context"]}\n\nTagged expressions: {tagged}'

    response = client.chat.completions.create(
        model=GPT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        temperature=0,
    )
    result = response.choices[0].message.content.strip()
    return None if result.upper() == "NONE" else result


def filter_clock_quotes(candidates: list[dict]) -> list[dict]:
    clock_quotes = []

    for item in tqdm(candidates, desc="GPT extraction"):
        excerpt = extract_clock_quote(item)

        if excerpt is not None:
            clock_quotes.append({
                "sentence_index": item["sentence_index"],
                "target_sentence": excerpt,
                "full_context": item["full_context"],
                "timex3_entities": item["timex3_entities"],
            })

    return clock_quotes

In [ ]:
'''
Filter the tagged sentences and write 
GPT's output to a JSON file.
'''
clock_quotes = filter_clock_quotes(candidates)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(clock_quotes, f, indent=4, ensure_ascii=False, cls=NumpyEncoder)

print(f"Saved {len(clock_quotes)} clock quotes to {OUTPUT_FILE}")